In [392]:
import pandas as pd

In [393]:
df = pd.read_csv("Customer_churn_Dataset.csv")

In [394]:
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,ContractType,InternetService,TotalCharges,TechSupport,Churn
0,1,49,Male,4,88.35,Month-to-Month,Fiber Optic,353.40,Yes,Yes
1,2,43,Male,0,36.67,Month-to-Month,Fiber Optic,0.00,Yes,Yes
2,3,51,Female,2,63.79,Month-to-Month,Fiber Optic,127.58,No,Yes
3,4,60,Female,8,102.34,One-Year,DSL,818.72,Yes,Yes
4,5,42,Male,32,69.01,Month-to-Month,NaN,2208.32,No,Yes


In [395]:
df.isnull().sum()

CustomerID           0
Age                  0
Gender               0
Tenure               0
MonthlyCharges       0
ContractType         0
InternetService    297
TotalCharges         0
TechSupport          0
Churn                0
dtype: int64

In [396]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       1000 non-null   int64  
 1   Age              1000 non-null   int64  
 2   Gender           1000 non-null   str    
 3   Tenure           1000 non-null   int64  
 4   MonthlyCharges   1000 non-null   float64
 5   ContractType     1000 non-null   str    
 6   InternetService  703 non-null    str    
 7   TotalCharges     1000 non-null   float64
 8   TechSupport      1000 non-null   str    
 9   Churn            1000 non-null   str    
dtypes: float64(2), int64(3), str(5)
memory usage: 104.5 KB


In [397]:
df.isnull().sum()

CustomerID           0
Age                  0
Gender               0
Tenure               0
MonthlyCharges       0
ContractType         0
InternetService    297
TotalCharges         0
TechSupport          0
Churn                0
dtype: int64

In [398]:
df.duplicated().sum()

0

In [399]:
df.describe() # numeric summary

,CustomerID,Age,Tenure,MonthlyCharges,TotalCharges
count,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000
mean,500.500000,44.674000,18.97300,74.391290,1404.364060
std,288.819436,9.797741,18.89257,25.712083,1571.755048
min,1.000000,12.000000,0.00000,30.000000,0.000000
25%,250.750000,38.000000,5.00000,52.357500,345.217500
50%,500.500000,45.000000,13.00000,74.060000,872.870000
75%,750.250000,51.000000,26.00000,96.102500,1900.175000
max,1000.000000,83.000000,122.00000,119.960000,12416.250000


In [400]:
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,ContractType,InternetService,TotalCharges,TechSupport,Churn
0,1,49,Male,4,88.35,Month-to-Month,Fiber Optic,353.40,Yes,Yes
1,2,43,Male,0,36.67,Month-to-Month,Fiber Optic,0.00,Yes,Yes
2,3,51,Female,2,63.79,Month-to-Month,Fiber Optic,127.58,No,Yes
3,4,60,Female,8,102.34,One-Year,DSL,818.72,Yes,Yes
4,5,42,Male,32,69.01,Month-to-Month,NaN,2208.32,No,Yes


### Selecting features:

In [401]:
from sklearn.preprocessing import LabelEncoder

In [402]:
numric_features = df.drop(columns=["CustomerID"]).describe().columns
categorical_features = df.drop(columns=["Churn"]).describe(include="object").columns
numric_features

C:\Users\KANHAIYA\AppData\Local\Temp\ipykernel_20472\1209244192.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.drop(columns=["Churn"]).describe(include="object").columns


Index(['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')

In [403]:
X = df.drop(columns = ["Churn"])
y = df["Churn"]

In [404]:
encoder = LabelEncoder()

In [405]:
y = encoder.fit_transform(y)

In [459]:
y[:6]

array([1, 1, 1, 1, 1, 1])

In [406]:
from sklearn.model_selection import train_test_split

In [407]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [408]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [409]:
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from xgboost import XGBClassifier

from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Ridge

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import joblib

In [410]:
categorical_transformer =Pipeline( 
    steps=[
    ("imputer",SimpleImputer(fill_value="Missing", strategy="constant")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
]
)

In [411]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# PIPELINES


## LogisticRegression

In [412]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("feature_select", SelectFromModel(Ridge())),
        ("classifier", LogisticRegression())
    ]
)

In [413]:
grid = {
    "preprocessor__num" : [StandardScaler(), MinMaxScaler()]
}

In [414]:
grid_search = GridSearchCV(pipe, grid,  cv=5, n_jobs=-1, scoring="roc_auc")

In [415]:
grid_search.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor__num': [StandardScaler(), MinMaxScaler()]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- 

In [416]:
logistic_model = grid_search.best_estimator_

In [417]:
y_pred = logistic_model.predict(X_test)

In [418]:
print("LOGISTIC REGRESSION")
print("📊 Comprehensive Classification Report:")
print("-" * 55)
print(classification_report(y_test, y_pred))

print("\n🧩 Confusion Matrix Layout:")
print("-" * 55)
cm = confusion_matrix(y_test, y_pred)
print(cm)

LOGISTIC REGRESSION
📊 Comprehensive Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

           0       0.87      0.57      0.68        23
           1       0.95      0.99      0.97       177

    accuracy                           0.94       200
   macro avg       0.91      0.78      0.83       200
weighted avg       0.94      0.94      0.93       200


🧩 Confusion Matrix Layout:
-------------------------------------------------------
[[ 13  10]
 [  2 175]]


In [419]:
joblib.dump(logistic_model,"logistic_model.pkl")

['logistic_model.pkl']

## RandomForestClassifier

In [420]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("feature_select", SelectFromModel(Ridge())),
        ("classifier", RandomForestClassifier())
    ]
)

In [421]:
grid = {
    "preprocessor__num" : [StandardScaler(), MinMaxScaler(),"passthrough"],
    "classifier__n_estimators" : [100, 200 ,300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__max_features': ['sqrt', 'log2']
}

In [422]:
random_search =  RandomizedSearchCV(pipe, grid, n_iter=15, n_jobs=-1, random_state=42, cv=5, scoring="roc_auc")

In [423]:
random_search.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__max_depth': [None, 10, ...], 'classifier__max_features': ['sqrt', 'log2'], 'classifier__n_estimators': [100, 200, ...], 'preprocessor__num': [StandardScaler(), MinMaxScaler(), ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` f

In [424]:
randomforest_classifier = random_search.best_estimator_

In [425]:
y_pred = randomforest_classifier.predict(X_test)

In [426]:
print("RandomForestClassifier")
print("📊 Comprehensive Classification Report:")
print("-" * 55)
print(classification_report(y_test, y_pred))

print("\n🧩 Confusion Matrix Layout:")
print("-" * 55)
cm = confusion_matrix(y_test, y_pred)
print(cm)

RandomForestClassifier
📊 Comprehensive Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

           0       0.85      1.00      0.92        23
           1       1.00      0.98      0.99       177

    accuracy                           0.98       200
   macro avg       0.93      0.99      0.95       200
weighted avg       0.98      0.98      0.98       200


🧩 Confusion Matrix Layout:
-------------------------------------------------------
[[ 23   0]
 [  4 173]]


In [427]:
joblib.dump(randomforest_classifier, "randomforest_classifier.pkl")

['randomforest_classifier.pkl']

## RidgeClassifier

In [428]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("feature_select", SelectFromModel(Ridge())),
        ("classifier", RidgeClassifier())
    ]
)

In [429]:
grid = {
    "preprocessor__num" : [StandardScaler(), MinMaxScaler(),"passthrough"],
    "classifier__alpha": [0.1, 1.0, 10.0, 100.0],
    "classifier__solver": ['auto', 'svd', 'cholesky', 'saga'],
    "classifier__class_weight": [None, 'balanced']

}

In [430]:
grid_search = GridSearchCV(pipe, grid, cv = 5, n_jobs=-1, scoring="roc_auc")

In [431]:
grid_search.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__alpha': [0.1, 1.0, ...], 'classifier__class_weight': [None, 'balanced'], 'classifier__solver': ['auto', 'svd', ...], 'preprocessor__num': [StandardScaler(), MinMaxScaler(), ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher,

In [432]:
ridge_Classifier = grid_search.best_estimator_

In [433]:
y_pred = ridge_Classifier.predict(X_test)

In [434]:
print("RidgeClassifier")
print("📊 Comprehensive Classification Report:")
print("-" * 55)
print(classification_report(y_test, y_pred))

print("\n🧩 Confusion Matrix Layout:")
print("-" * 55)
cm = confusion_matrix(y_test, y_pred)
print(cm)

RidgeClassifier
📊 Comprehensive Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

           0       0.44      1.00      0.61        23
           1       1.00      0.84      0.91       177

    accuracy                           0.85       200
   macro avg       0.72      0.92      0.76       200
weighted avg       0.94      0.85      0.88       200


🧩 Confusion Matrix Layout:
-------------------------------------------------------
[[ 23   0]
 [ 29 148]]


In [435]:
joblib.dump(ridge_Classifier, "ridge_Classifier.pkl")

['ridge_Classifier.pkl']

## XGBoost

In [436]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("feature_select", SelectFromModel(Ridge())),
        ("classifier", XGBClassifier())
    ]
)

In [437]:
grid = {
    "preprocessor__num" : [StandardScaler(), MinMaxScaler(),"passthrough"],
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7, 9]
}

In [438]:
random_search = RandomizedSearchCV(pipe, grid, n_iter=20, cv= 5, n_jobs=-1, random_state=42, scoring="roc_auc")

In [439]:
random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__learning_rate': [0.01, 0.05, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [100, 200, ...], 'preprocessor__num': [StandardScaler(), MinMaxScaler(), ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` f

In [440]:
xgb_classifier = random_search.best_estimator_

In [441]:
y_pred = xgb_classifier.predict(X_test)

In [442]:
print("XGBClassifier")
print("📊 Comprehensive Classification Report:")
print("-" * 55)
print(classification_report(y_test, y_pred))

print("\n🧩 Confusion Matrix Layout:")
print("-" * 55)
cm = confusion_matrix(y_test, y_pred)
print(cm)

XGBClassifier
📊 Comprehensive Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        23
           1       1.00      0.99      0.99       177

    accuracy                           0.99       200
   macro avg       0.96      0.99      0.98       200
weighted avg       0.99      0.99      0.99       200


🧩 Confusion Matrix Layout:
-------------------------------------------------------
[[ 23   0]
 [  2 175]]


In [443]:
joblib.dump(xgb_classifier, "xgb_classifier.pkl")

['xgb_classifier.pkl']

# BestClassifier

In [444]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("feature_select", SelectFromModel(Ridge())),
        ("classifier", LogisticRegression()) # default
    ]
)

In [445]:
grid = [
    {
    "classifier" : [LogisticRegression()],
    "preprocessor__num" : [StandardScaler(), MinMaxScaler()]
},
    {
    "classifier" : [XGBClassifier()],
    "preprocessor__num" : [StandardScaler(), MinMaxScaler(),"passthrough"],
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7, 9]
},
    {
    "classifier" : [RidgeClassifier()],
    "preprocessor__num" : [StandardScaler(), MinMaxScaler(),"passthrough"],
    "classifier__alpha": [0.1, 1.0, 10.0, 100.0],
    "classifier__solver": ['auto', 'svd', 'cholesky', 'saga'],
    "classifier__class_weight": [None, 'balanced']

},
    {
    "classifier" : [RandomForestClassifier()],
    "preprocessor__num" : [StandardScaler(), MinMaxScaler(),"passthrough"],
    "classifier__n_estimators" : [100, 200 ,300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__max_features': ['sqrt', 'log2']
}
]

In [446]:
random_search = RandomizedSearchCV(pipe, grid, n_iter=50, cv =5, n_jobs=-1, random_state=42, scoring="roc_auc")

In [447]:
random_search.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'classifier': [LogisticRegression()], 'preprocessor__num': [StandardScaler(), MinMaxScaler()]}, {'classifier': [XGBClassifier...ree=None, ...)], 'classifier__learning_rate': [0.01, 0.05, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [100, 200, ...], ...}, ...]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiat

In [448]:
best_classifier = random_search.best_estimator_

In [449]:
y_pred = best_classifier.predict(X_test)

In [450]:
print("BestClassifier")
print("📊 Comprehensive Classification Report:")
print("-" * 55)
print(classification_report(y_test, y_pred))

print("\n🧩 Confusion Matrix Layout:")
print("-" * 55)
cm = confusion_matrix(y_test, y_pred)
print(cm)

BestClassifier
📊 Comprehensive Classification Report:
-------------------------------------------------------
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        23
           1       1.00      0.99      1.00       177

    accuracy                           0.99       200
   macro avg       0.98      1.00      0.99       200
weighted avg       1.00      0.99      1.00       200


🧩 Confusion Matrix Layout:
-------------------------------------------------------
[[ 23   0]
 [  1 176]]


In [451]:
joblib.dump(best_classifier,"best_classifier.pkl")

['best_classifier.pkl']

## Selected features:

In [453]:
# 1. Get all encoded feature names from the preprocessor step
all_features = best_classifier.named_steps['preprocessor'].get_feature_names_out()

# 2. Extract the True/False mask from the feature selection step
# (This tells us which indices were kept by Ridge)
selection_mask = best_classifier.named_steps['feature_select'].get_support()

# 3. Filter the original names using the mask
selected_feature_names = all_features[selection_mask]

In [454]:
selected_feature_names

array(['num__Tenure', 'num__TotalCharges',
       'cat__ContractType_Month-to-Month', 'cat__ContractType_One-Year',
       'cat__ContractType_Two-Year', 'cat__TechSupport_No',
       'cat__TechSupport_Yes'], dtype=object)

In [460]:
df.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,ContractType,InternetService,TotalCharges,TechSupport,Churn
0,1,49,Male,4,88.35,Month-to-Month,Fiber Optic,353.40,Yes,Yes
1,2,43,Male,0,36.67,Month-to-Month,Fiber Optic,0.00,Yes,Yes
2,3,51,Female,2,63.79,Month-to-Month,Fiber Optic,127.58,No,Yes
3,4,60,Female,8,102.34,One-Year,DSL,818.72,Yes,Yes
4,5,42,Male,32,69.01,Month-to-Month,NaN,2208.32,No,Yes
